# Step 16 — Export held-out predictions for the app

This notebook creates a static app artifact containing momentum-only and tone-only predictions for the fixed held-out period. The Streamlit app reads these completed results and does not retrain models.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_sentiment.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "explorer_predictions.csv"
data = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").sort_index().loc[:"2024-11-22"]
test_size = int(len(data) * 0.25)
test_start = len(data) - test_size
train = data.iloc[: test_start - 5].copy()
test = data.iloc[test_start:].copy()
y_train = train["outperformed"].astype(int)

output = test.copy()
feature_sets = {
    "momentum": ["aapl_momentum_5d"],
    "tone": ["average_sentiment_5d"],
}
for name, features in feature_sets.items():
    model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
    model.fit(train[features], y_train)
    output[f"{name}_probability"] = model.predict_proba(test[features])[:, 1]
    output[f"{name}_prediction"] = model.predict(test[features])
    output[f"{name}_correct"] = output[f"{name}_prediction"].eq(output["outperformed"])

output.index.name = "date"
output.to_csv(OUTPUT_PATH)
print(f"Saved {len(output)} held-out rows to {OUTPUT_PATH}")
output.head()

Saved 118 held-out rows to /Users/keishakalra/Desktop/Financial_App/data/processed/explorer_predictions.csv


,aapl_adjusted_close,spy_adjusted_close,aapl_momentum_5d,aapl_future_return_5d,spy_future_return_5d,outperformed,average_sentiment_5d,headline_count_5d,momentum_probability,momentum_prediction,momentum_correct,tone_probability,tone_prediction,tone_correct
date,,,,,,,,,,,,,,
2024-06-07,195.199097,519.982971,0.024135,0.079232,0.016423,1,0.232461,34,0.515866,1,True,0.478921,0,False
2024-06-10,191.461472,521.589539,-0.004690,0.121945,0.021357,1,0.236392,44,0.534305,1,True,0.478260,0,False
2024-06-11,205.370987,522.845764,0.065861,0.034468,0.021492,1,0.219777,61,0.489110,0,False,0.481053,0,False
2024-06-12,211.240143,527.139893,0.087813,-0.015910,0.010418,0,0.248154,73,0.475049,0,True,0.476284,0,True
2024-06-13,212.400085,528.201294,0.101604,-0.031507,0.007036,0,0.219026,77,0.466234,0,True,0.481180,0,True
